# SinLlama 1B QA Fine-Tuning v2

This notebook improves grounded Sinhala question answering while retaining the Sinhala continual-pretraining setup from `sinllama_1b_cpt.ipynb`.

Key changes:

- Uses the tokenizer saved with the Sinhala CPT adapter and merges that adapter before QA training.
- Converts every unanswerable target to one canonical refusal sentence.
- Uses answer-aware evidence windows so long contexts do not truncate away the answer.
- Splits validation data by context group to prevent context leakage.
- Uses prompt/completion records and completion-only loss, so training loss is applied only to answers.
- Uses deterministic inference, top evidence-window retrieval, and a lexical grounding gate.
- Reports honest normalized exact match, token F1, and unanswerable false-answer rate.

The external test file is used only after training and never for model selection.

In [ ]:
%uv pip install -q "transformers>=4.51,<5" "trl>=0.26,<0.30" "peft>=0.19" datasets accelerate sentencepiece huggingface_hub hf_transfer scikit-learn tqdm

In [ ]:
import hashlib
import json
import math
import os
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from huggingface_hub import login
from tqdm.auto import tqdm
from transformers import set_seed

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

SEED = 42
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B"
CPT_ADAPTER_ID = "isji/sinllama-1b-cpt"

TRAIN_CANDIDATES = [
    Path("/tmp/merged_train.jsonl"),
    Path("/tmp/train.jsonl"),
    Path("splits/train.jsonl"),
]
TRAIN_PATH = next((path for path in TRAIN_CANDIDATES if path.is_file()), TRAIN_CANDIDATES[0])
TEST_PATH = Path("/tmp/test_updated.jsonl")

OUTPUT_DIR = Path("/tmp/output_sinllama_1b_qa_v2")
QA_ADAPTER_DIR = Path("/tmp/sinllama_1b_qa_v2_adapter")
MERGED_MODEL_DIR = Path("/tmp/sinllama_1b_qa_v2_merged")
RESULTS_PATH = Path("/tmp/sinllama_1b_qa_v2_results.jsonl")
MERGED_REPO_ID = os.environ.get("QA_MERGED_REPO_ID", "isji/sinllama-1b-qa-v2-merged")
HF_REPO_PRIVATE = True

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 48
VALIDATION_FRACTION = 0.02
MIN_UNANSWERABLE_TRAIN_FRACTION = 0.25
TOP_K_WINDOWS = 2
GROUNDING_THRESHOLD = 0.50

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this notebook.")
if "HF_TOKEN" not in os.environ:
    raise RuntimeError("Add HF_TOKEN as a Modal secret; never paste it into the notebook.")

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("GPU:", torch.cuda.get_device_name(0))
print("Training data:", TRAIN_PATH)
print("External test data:", TEST_PATH)

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

train_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Use the tokenizer stored with the CPT adapter so its vocabulary IDs exactly match CPT training.
tokenizer = AutoTokenizer.from_pretrained(CPT_ADAPTER_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"

# Confirm the extended tokenizer preserves every original Llama token ID.
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
id_mismatches = sum(
    tokenizer.convert_tokens_to_ids(token) != token_id
    for token, token_id in base_tokenizer.get_vocab().items()
)
if id_mismatches:
    raise RuntimeError(f"Extended tokenizer remapped {id_mismatches:,} base token IDs.")
del base_tokenizer

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=train_dtype,
    device_map={"": torch.cuda.current_device()},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)

# The CPT notebook trained an expanded Sinhala vocabulary. Resize before loading its adapter.
model.resize_token_embeddings(len(tokenizer))
print("Loading and merging Sinhala CPT adapter...")
model = PeftModel.from_pretrained(model, CPT_ADAPTER_ID)
model = model.merge_and_unload()
if model.get_input_embeddings().num_embeddings != len(tokenizer):
    raise RuntimeError("Input embedding vocabulary does not match the CPT tokenizer.")
if model.get_output_embeddings().out_features != len(tokenizer):
    raise RuntimeError("Output head vocabulary does not match the CPT tokenizer.")
print("Tied input/output embeddings:", model.config.tie_word_embeddings)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

print("Tokenizer size:", len(tokenizer))
print("CPT model merged and ready for QA LoRA training.")

In [ ]:
INSTRUCTION = f"""උපදෙස්: පහත සන්දර්භය පමණක් භාවිතා කර ප්‍රශ්නයට පිළිතුරු දෙන්න.
- පිළිතුර සන්දර්භයේ තිබේ නම්, එයින් කෙටිම නිශ්චිත වචන පෙළ පමණක් දෙන්න.
- අමතර පැහැදිලි කිරීම්, පිටත දැනුම හෝ අනුමාන එකතු නොකරන්න.
- පිළිතුර සන්දර්භයේ පැහැදිලිව නොමැති නම්, හරියටම මෙය පමණක් දෙන්න: {NO_ANSWER}"""

SINHALA_WORD_RE = re.compile(r"[\w\u0D80-\u0DFF]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def build_prompt(context, question):
    return (
        f"{INSTRUCTION}\n\n"
        f"සන්දර්භය:\n{clean_text(context)}\n\n"
        f"ප්‍රශ්නය:\n{clean_text(question)}\n\n"
        "පිළිතුර:\n"
    )


def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSONL file not found: {path}")

    records = []
    fingerprints = set()
    dropped = 0
    duplicates = 0

    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error

            question = clean_text(item.get("question"))
            context = clean_text(item.get("context"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))

            normalized = {
                "question": question,
                "context": context,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
            }
            if not question or not context or (answerable and not normalized["answer"]):
                dropped += 1
                continue

            fingerprint = (
                normalized["question"],
                normalized["context"],
                normalized["answer"],
                normalized["answerable"],
            )
            if fingerprint in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fingerprint)
            records.append(normalized)

    print(f"Loaded {len(records):,} unique records from {path}")
    print(f"Dropped invalid/empty: {dropped:,}; exact duplicates removed: {duplicates:,}")
    return records


records = load_jsonl(TRAIN_PATH)
answerable_count = sum(item["answerable"] for item in records)
print(f"Answerable: {answerable_count:,}; unanswerable: {len(records) - answerable_count:,}")

In [ ]:
def token_supported(token, normalized_context):
    if token in normalized_context:
        return True
    # Sinhala case endings often add one character; a short stem check preserves grounded variants.
    return len(token) >= 4 and token[:-1] in normalized_context


def rank_context_windows(context, query, token_budget, top_k=1):
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    if len(context_ids) <= token_budget:
        return [(context, 1.0)]

    query_tokens = set(lexical_tokens(query))
    stride = max(64, token_budget // 2)
    candidates = []

    for start in range(0, len(context_ids), stride):
        chunk_ids = context_ids[start : start + token_budget]
        if len(chunk_ids) < 32:
            continue
        chunk = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()
        normalized_chunk = " ".join(lexical_tokens(chunk))
        if query_tokens:
            matched = sum(token_supported(token, normalized_chunk) for token in query_tokens)
            score = matched / len(query_tokens)
        else:
            score = 0.0
        candidates.append((chunk, score, start))
        if start + token_budget >= len(context_ids):
            break

    candidates.sort(key=lambda value: (-value[1], value[2]))
    return [(chunk, score) for chunk, score, _ in candidates[:top_k]]


def context_budget(question, completion):
    fixed_tokens = len(tokenizer(build_prompt("", question), add_special_tokens=False)["input_ids"])
    completion_tokens = len(tokenizer(completion, add_special_tokens=False)["input_ids"])
    return max(128, MAX_LENGTH - fixed_tokens - completion_tokens - 24)


def make_training_example(item):
    answer = canonical_answer(item)
    completion = answer + tokenizer.eos_token
    budget = context_budget(item["question"], completion)
    query = item["question"] if not item["answerable"] else f"{item['question']} {answer}"
    window = rank_context_windows(item["context"], query, budget, top_k=1)[0][0]
    prompt = build_prompt(window, item["question"])

    # Keep a safety loop so TRL's right truncation can never remove completion supervision.
    for _ in range(2):
        total_length = len(tokenizer(prompt + completion, add_special_tokens=False)["input_ids"])
        if total_length <= MAX_LENGTH - 4:
            break
        budget = max(128, budget - (total_length - MAX_LENGTH) - 16)
        window = rank_context_windows(item["context"], query, budget, top_k=1)[0][0]
        prompt = build_prompt(window, item["question"])

    return {"prompt": prompt, "completion": completion}


# Split on context hashes, not individual rows, to prevent the same passage leaking into validation.
groups = {}
for item in records:
    key = hashlib.sha1(item["context"].encode("utf-8")).hexdigest()
    groups.setdefault(key, []).append(item)

group_keys = sorted(groups)
random.Random(SEED).shuffle(group_keys)
# Use at least 200 validation contexts for small datasets, capped at 20%.
desired_validation_groups = max(200, round(len(group_keys) * VALIDATION_FRACTION))
validation_group_cap = max(1, math.floor(len(group_keys) * 0.20))
validation_group_count = min(
    desired_validation_groups,
    validation_group_cap,
    max(1, len(group_keys) - 1),
)
validation_keys = set(group_keys[:validation_group_count])

train_records = []
validation_records = []
for key, group in groups.items():
    (validation_records if key in validation_keys else train_records).extend(group)

def rebalance_unanswerable(items, minimum_fraction):
    positives = [item for item in items if item["answerable"]]
    negatives = [item for item in items if not item["answerable"]]
    if not negatives or len(negatives) / len(items) >= minimum_fraction:
        return list(items), 0

    required_negative_count = math.ceil(
        minimum_fraction * len(positives) / (1.0 - minimum_fraction)
    )
    extra_count = max(0, required_negative_count - len(negatives))
    rng = random.Random(SEED)
    balanced = list(items) + [rng.choice(negatives) for _ in range(extra_count)]
    rng.shuffle(balanced)
    return balanced, extra_count


train_records, repeated_negative_count = rebalance_unanswerable(
    train_records,
    MIN_UNANSWERABLE_TRAIN_FRACTION,
)

print(f"Train records after balancing: {len(train_records):,}")
print(f"Repeated hard-negative rows added: {repeated_negative_count:,}")
print(f"Validation records: {len(validation_records):,}")
print("Preparing answer-preserving prompt/completion examples...")

train_examples = [make_training_example(item) for item in tqdm(train_records, desc="Train")]
validation_examples = [make_training_example(item) for item in tqdm(validation_records, desc="Validation")]

train_dataset = Dataset.from_list(train_examples)
validation_dataset = Dataset.from_list(validation_examples)

# Audit a sample after preprocessing.
audit_sample = random.Random(SEED).sample(
    train_examples,
    min(1000, len(train_examples)),
)
audit_lengths = [
    len(tokenizer(item["prompt"] + item["completion"], add_special_tokens=False)["input_ids"])
    for item in audit_sample
]
print(f"Audited maximum sequence length: {max(audit_lengths):,}/{MAX_LENGTH:,}")
print("\n--- Prepared example ---")
print(train_dataset[0]["prompt"])
print(train_dataset[0]["completion"])

In [ ]:
from peft import LoraConfig, TaskType
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

qa_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

training_epochs = 2 if len(train_dataset) >= 50_000 else 5
print(f"Training epochs selected for dataset size: {training_epochs}")

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    eval_packing=False,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    learning_rate=8e-5,
    num_train_epochs=training_epochs,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=(train_dtype == torch.bfloat16),
    fp16=(train_dtype == torch.float16),
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    logging_first_step=True,
    group_by_length=True,
    dataset_num_proc=min(8, os.cpu_count() or 1),
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=qa_lora_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.model.print_trainable_parameters()
print("Starting grounded Sinhala QA fine-tuning...")
train_result = trainer.train()
print(train_result)

In [ ]:
# Save the lightweight QA adapter first.
QA_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(QA_ADAPTER_DIR))
tokenizer.save_pretrained(QA_ADAPTER_DIR)

# Also save one deployable model containing both the Sinhala CPT and QA LoRA changes.
print("Merging the trained QA adapter into the CPT-merged model...")
model = trainer.model.merge_and_unload()
model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.eval()

MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(
    MERGED_MODEL_DIR,
    safe_serialization=True,
    max_shard_size="2GB",
)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("QA adapter:", QA_ADAPTER_DIR)
print("Deployable merged model:", MERGED_MODEL_DIR)

In [ ]:
def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


def evidence_support(answer, context):
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))
    supported = sum(token_supported(token, normalized_context) for token in answer_tokens)
    return supported / len(answer_tokens)


def generate_candidate(context_window, question):
    prompt = build_prompt(context_window, question)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1] :]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.splitlines()[0].strip(" []{}()<>\"'`") if answer else ""
    return answer


def run_qa(context, question, use_grounding=True):
    completion_stub = NO_ANSWER + tokenizer.eos_token
    budget = context_budget(question, completion_stub)
    windows = rank_context_windows(context, question, budget, top_k=TOP_K_WINDOWS)
    candidates = []

    for window, retrieval_score in windows:
        raw_answer = generate_candidate(window, question)
        support = 1.0 if is_no_answer(raw_answer) else evidence_support(raw_answer, window)
        candidates.append({
            "raw_answer": raw_answer,
            "window": window,
            "retrieval_score": retrieval_score,
            "support": support,
        })

    grounded = [
        candidate for candidate in candidates
        if candidate["raw_answer"] and not is_no_answer(candidate["raw_answer"])
        and candidate["support"] >= GROUNDING_THRESHOLD
    ]

    if grounded:
        best = max(
            grounded,
            key=lambda candidate: (candidate["support"], candidate["retrieval_score"]),
        )
        final_answer = best["raw_answer"]
    else:
        best = max(candidates, key=lambda candidate: candidate["retrieval_score"])
        final_answer = NO_ANSWER if use_grounding else best["raw_answer"]

    return {
        "answer": final_answer,
        "raw_answer": best["raw_answer"],
        "support": best["support"],
        "retrieval_score": best["retrieval_score"],
        "candidate_count": len(candidates),
    }


print("Grounded run_qa(context, question) is ready.")

In [ ]:
def token_f1(prediction, reference):
    prediction_digits = re.findall(r"\d+", normalize_answer(prediction))
    reference_digits = re.findall(r"\d+", normalize_answer(reference))
    if reference_digits and prediction_digits != reference_digits:
        return 0.0
    prediction_tokens = lexical_tokens(prediction)
    reference_tokens = lexical_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0
    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


if not TEST_PATH.is_file():
    print(f"Skipping external evaluation because {TEST_PATH} is not present.")
else:
    test_records = load_jsonl(TEST_PATH)
    exact_correct = 0
    raw_exact_correct = 0
    f1_total = 0.0
    answerable_correct = 0
    answerable_total = 0
    unanswerable_correct = 0
    unanswerable_total = 0
    unsupported_rejections = 0
    predicted_no_answer_count = 0
    correct_no_answer_count = 0

    with RESULTS_PATH.open("w", encoding="utf-8", newline="\n") as results_file:
        for index, item in enumerate(test_records, 1):
            reference = canonical_answer(item)
            result = run_qa(item["context"], item["question"], use_grounding=True)
            prediction = result["answer"]
            raw_prediction = result["raw_answer"]

            exact = normalize_answer(prediction) == normalize_answer(reference)
            raw_exact = normalize_answer(raw_prediction) == normalize_answer(reference)
            f1 = token_f1(prediction, reference)
            predicted_no_answer = is_no_answer(prediction)
            exact_correct += int(exact)
            raw_exact_correct += int(raw_exact)
            f1_total += f1
            predicted_no_answer_count += int(predicted_no_answer)
            correct_no_answer_count += int(predicted_no_answer and not item["answerable"])
            unsupported_rejections += int(
                prediction == NO_ANSWER and raw_prediction and not is_no_answer(raw_prediction)
            )

            if item["answerable"]:
                answerable_total += 1
                answerable_correct += int(exact)
            else:
                unanswerable_total += 1
                unanswerable_correct += int(exact)

            saved = {
                "index": index,
                "question": item["question"],
                "reference": reference,
                "raw_prediction": raw_prediction,
                "prediction": prediction,
                "answerable": item["answerable"],
                "exact_match": exact,
                "token_f1": f1,
                "evidence_support": result["support"],
                "retrieval_score": result["retrieval_score"],
            }
            results_file.write(json.dumps(saved, ensure_ascii=False) + "\n")
            results_file.flush()

            print("\n" + "=" * 100, flush=True)
            print(f"[{index}/{len(test_records)}]", flush=True)
            print("Question :", item["question"], flush=True)
            print("Reference:", reference, flush=True)
            print("Raw      :", raw_prediction, flush=True)
            print("Final    :", prediction, flush=True)
            print(f"Support  : {result['support']:.3f}", flush=True)
            print(f"Exact/F1 : {exact} / {f1:.3f}", flush=True)

    total = len(test_records)
    print("\n" + "=" * 100)
    print("EXTERNAL TEST RESULTS")
    print("=" * 100)
    print(f"Grounded exact match : {exact_correct}/{total} ({100 * exact_correct / total:.2f}%)")
    print(f"Raw exact match      : {raw_exact_correct}/{total} ({100 * raw_exact_correct / total:.2f}%)")
    print(f"Mean token F1        : {f1_total / total:.4f}")
    print(f"Answerable exact     : {answerable_correct}/{answerable_total}")
    print(f"Unanswerable exact   : {unanswerable_correct}/{unanswerable_total}")
    false_answers = unanswerable_total - unanswerable_correct
    no_answer_precision = correct_no_answer_count / max(predicted_no_answer_count, 1)
    no_answer_recall = correct_no_answer_count / max(unanswerable_total, 1)
    no_answer_f1 = (
        2 * no_answer_precision * no_answer_recall / (no_answer_precision + no_answer_recall)
        if no_answer_precision + no_answer_recall else 0.0
    )
    print(f"False-answer rate on unanswerable: {false_answers}/{unanswerable_total} "
          f"({100 * false_answers / max(unanswerable_total, 1):.2f}%)")
    print(f"No-answer precision/recall/F1: {no_answer_precision:.4f} / "
          f"{no_answer_recall:.4f} / {no_answer_f1:.4f}")
    print(f"Unsupported generations rejected: {unsupported_rejections}")
    print("Detailed results:", RESULTS_PATH)

## Push the complete QA model to Hugging Face

This uploads the independently loadable model containing the base model, Sinhala CPT changes, and QA changes. The matching CPT tokenizer is uploaded to the same repository. Set the optional Modal environment variable `QA_MERGED_REPO_ID` to override the default repository name.

In [ ]:
print(f"Pushing complete merged QA model to: {MERGED_REPO_ID}")
model.push_to_hub(
    MERGED_REPO_ID,
    token=os.environ["HF_TOKEN"],
    safe_serialization=True,
    max_shard_size="5GB",
    private=HF_REPO_PRIVATE,
    commit_message="Upload grounded Sinhala QA v2 merged model",
)
tokenizer.push_to_hub(
    MERGED_REPO_ID,
    token=os.environ["HF_TOKEN"],
    private=HF_REPO_PRIVATE,
    commit_message="Upload matching Sinhala CPT tokenizer",
)

print(f"Merged model pushed to https://huggingface.co/{MERGED_REPO_ID}")